# Correlação entre Depressão Adolescente e Uso de Redes Sociais nos EUA (2011–2023)

**Estratégia A: CDC YRBS + Pew Research Center**

---

| Item | Detalhe |
|------|--------|
| **Autor** | [Seu nome] |
| **Orientador** | [Nome do orientador] |
| **Instituição** | [Sua instituição] |
| **População** | Adolescentes 13–17 anos, EUA |
| **Período** | 2011–2023 |
| **Fontes** | CDC YRBS (depressão) + Pew Research (redes sociais) |
| **Tipo de análise** | Correlação de séries temporais + análise intra-survey |

---

### Problema de pesquisa

> *"Existe correlação estatisticamente significativa entre o aumento do uso de redes sociais e o aumento na prevalência de sintomas depressivos entre adolescentes norte-americanos (13–17 anos) no período de 2011 a 2023?"*

### Estrutura deste notebook

1. **Setup e dependências**
2. **Parte A — Análise de séries temporais** (dados agregados, 2011–2023)
   - 2.1 Construção dos datasets
   - 2.2 Análise exploratória (EDA)
   - 2.3 Interpolação e alinhamento temporal
   - 2.4 Visualização de tendências
   - 2.5 Análise de correlação
   - 2.6 Visualização da correlação
   - 2.7 Estratificação por sexo
   - 2.8 Gap de gênero
3. **Parte B — Análise intra-survey** (microdados YRBS 2023)
   - 3.1 Carregamento dos microdados
   - 3.2 Análise exploratória dos microdados
   - 3.3 Tabela cruzada e testes estatísticos
   - 3.4 Visualizações
4. **Síntese e conclusões**

---
## 1. Setup e Dependências

Importação das bibliotecas utilizadas em todo o notebook. Todas estão pré-instaladas no Google Colab.

- **pandas / numpy**: manipulação e estruturação de dados
- **scipy.stats**: testes estatísticos (Pearson, Spearman, qui-quadrado)
- **matplotlib / seaborn**: visualização de dados

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.lines import Line2D
import urllib.request
import os
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 7),
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
})

COLORS = {
    'depression': '#C0392B',
    'depression_f': '#E74C3C',
    'depression_m': '#3498DB',
    'social_media': '#2980B9',
    'accent': '#1F4E79',
    'grid': '#E8E8E8',
    'bg': '#FAFAFA',
    'freq_sm': '#E74C3C',
    'infreq_sm': '#3498DB',
}

print('✅ Setup concluído — todas as bibliotecas carregadas')

---
## 2. PARTE A — Análise de Séries Temporais (2011–2023)

Nesta seção, cruzamos dados agregados de duas fontes independentes para avaliar se o aumento do uso de redes sociais acompanha o aumento de sintomas depressivos entre adolescentes nos EUA.

- **Eixo 1 (Depressão):** CDC Youth Risk Behavior Survey (YRBS) — pergunta Q26: *"During the past 12 months, did you ever feel so sad or hopeless almost every day for two weeks or more in a row that you stopped doing some usual activities?"*
- **Eixo 2 (Redes sociais):** Pew Research Center — porcentagem de adolescentes que dizem estar online *"almost constantly"*

### 2.1 Construção dos datasets

Os dados abaixo foram extraídos manualmente dos relatórios oficiais de cada instituição. O YRBS é bienal (anos ímpares); o Pew publica em anos irregulares. As referências completas estão na seção final deste notebook.

In [ ]:
# Eixo 1: Depressão — CDC YRBS
# Fonte: YRBS Data Summary & Trends Report 2013–2023 (CDC, 2024)
yrbs = pd.DataFrame({
    'ano':                [2011,  2013,  2015,  2017,  2019,  2021,  2023],
    'depressao_overall':  [28.5,  29.9,  29.9,  31.5,  36.7,  42.3,  39.7],
    'depressao_feminino': [35.9,  39.1,  39.8,  41.1,  46.6,  57.0,  53.0],
    'depressao_masculino':[21.5,  20.8,  20.3,  21.4,  26.1,  28.5,  28.0],
})

print('Eixo 1 — % de adolescentes com tristeza/desesperança persistente (YRBS):')
display(yrbs)

In [ ]:
# Eixo 2: Redes sociais — Pew Research Center
# Fonte: Pew "Teens, Social Media and Technology" (2018, 2022, 2023, 2024)
pew = pd.DataFrame({
    'ano':               [2015,  2018,  2022,  2023,  2024],
    'online_constantly': [24.0,  45.0,  46.0,  46.0,  46.0],
    'usam_social_media': [76.0,  89.0,  95.0,  93.0,  90.0],
})

print('Eixo 2 — % de adolescentes por padrão de uso de redes sociais (Pew):')
display(pew)

### 2.2 Análise Exploratória de Dados (EDA) — Séries Temporais

Antes de prosseguir com a correlação, inspecionamos as características básicas de cada série: estatísticas descritivas, valores ausentes, e a magnitude da variação ao longo do período.

In [ ]:
print('═' * 60)
print('  EDA — EIXO 1: DEPRESSÃO (YRBS)')
print('═' * 60)
print(f'\n  Período: {yrbs["ano"].min()} a {yrbs["ano"].max()}')
print(f'  Pontos de dados: {len(yrbs)} (bienal)')
print(f'  Valores ausentes: {yrbs.isnull().sum().sum()}')
print()
print('  Estatísticas descritivas (% com tristeza/desesperança):')
print(f'    Overall  — Mín: {yrbs["depressao_overall"].min():.1f}%  Máx: {yrbs["depressao_overall"].max():.1f}%  Média: {yrbs["depressao_overall"].mean():.1f}%  DP: {yrbs["depressao_overall"].std():.1f}')
print(f'    Feminino — Mín: {yrbs["depressao_feminino"].min():.1f}%  Máx: {yrbs["depressao_feminino"].max():.1f}%  Média: {yrbs["depressao_feminino"].mean():.1f}%  DP: {yrbs["depressao_feminino"].std():.1f}')
print(f'    Masculino— Mín: {yrbs["depressao_masculino"].min():.1f}%  Máx: {yrbs["depressao_masculino"].max():.1f}%  Média: {yrbs["depressao_masculino"].mean():.1f}%  DP: {yrbs["depressao_masculino"].std():.1f}')

var_overall = ((yrbs['depressao_overall'].iloc[-1] - yrbs['depressao_overall'].iloc[0]) / yrbs['depressao_overall'].iloc[0]) * 100
var_fem = ((yrbs['depressao_feminino'].iloc[-1] - yrbs['depressao_feminino'].iloc[0]) / yrbs['depressao_feminino'].iloc[0]) * 100
var_masc = ((yrbs['depressao_masculino'].iloc[-1] - yrbs['depressao_masculino'].iloc[0]) / yrbs['depressao_masculino'].iloc[0]) * 100

print(f'\n  Variação no período ({yrbs["ano"].min()}–{yrbs["ano"].max()}):')
print(f'    Overall:  {var_overall:+.1f}%')
print(f'    Feminino: {var_fem:+.1f}%')
print(f'    Masculino:{var_masc:+.1f}%')

gap_inicio = yrbs['depressao_feminino'].iloc[0] - yrbs['depressao_masculino'].iloc[0]
gap_fim = yrbs['depressao_feminino'].iloc[-1] - yrbs['depressao_masculino'].iloc[-1]
print(f'\n  Gap de gênero (feminino - masculino):')
print(f'    {yrbs["ano"].iloc[0]}: {gap_inicio:.1f} pp')
print(f'    {yrbs["ano"].iloc[-1]}: {gap_fim:.1f} pp')
print(f'    Crescimento do gap: {gap_fim - gap_inicio:+.1f} pp')

In [ ]:
print('═' * 60)
print('  EDA — EIXO 2: REDES SOCIAIS (PEW)')
print('═' * 60)
print(f'\n  Período: {pew["ano"].min()} a {pew["ano"].max()}')
print(f'  Pontos de dados: {len(pew)}')
print(f'  Valores ausentes: {pew.isnull().sum().sum()}')
print()
print('  Estatísticas descritivas:')
print(f'    Online constantemente — Mín: {pew["online_constantly"].min():.0f}%  Máx: {pew["online_constantly"].max():.0f}%  Média: {pew["online_constantly"].mean():.1f}%')
print(f'    Usam social media     — Mín: {pew["usam_social_media"].min():.0f}%  Máx: {pew["usam_social_media"].max():.0f}%  Média: {pew["usam_social_media"].mean():.1f}%')

var_online = ((pew['online_constantly'].iloc[-1] - pew['online_constantly'].iloc[0]) / pew['online_constantly'].iloc[0]) * 100
print(f'\n  Variação "online constantemente" ({pew["ano"].iloc[0]}–{pew["ano"].iloc[-1]}): {var_online:+.1f}%')
print(f'\n  Observação: o salto mais expressivo ocorreu entre 2015 (24%)')
print(f'  e 2018 (45%), coincidindo com a adoção massiva de smartphones.')
print(f'  A partir de 2022, o indicador estabilizou em ~46%.')

**Achados da EDA (Parte A):**

- A prevalência de depressão subiu consistentemente de 2011 a 2021, com leve recuo em 2023. A variação acumulada foi de ~39% no geral, ~48% entre meninas, e ~30% entre meninos.
- O gap de gênero na depressão quase dobrou no período, passando de ~14pp para ~25pp.
- O uso constante de redes sociais praticamente dobrou entre 2015 e 2018, mas estabilizou a partir de 2022.
- Ambas as séries são monotonicamente crescentes na maior parte do período, o que favorece correlação positiva mas também exige cautela (duas tendências ascendentes podem correlacionar por acaso).
- Nenhum valor ausente em nenhuma das fontes.

### 2.3 Interpolação e alinhamento temporal

O YRBS coleta dados em anos ímpares; o Pew em anos irregulares. Para calcular correlação, precisamos de pares alinhados. A estratégia adotada é **interpolação linear**, que assume variação uniforme entre os pontos de dados reais. Esta premissa é documentada como limitação.

In [ ]:
anos_completos = list(range(2011, 2025))

# Interpolar YRBS
yrbs_interp = pd.DataFrame({'ano': anos_completos})
yrbs_interp = yrbs_interp.merge(yrbs[['ano','depressao_overall','depressao_feminino','depressao_masculino']], on='ano', how='left')
yrbs_interp = yrbs_interp.set_index('ano').interpolate(method='linear').reset_index()

# Interpolar Pew
pew_interp = pd.DataFrame({'ano': anos_completos})
pew_interp = pew_interp.merge(pew[['ano','online_constantly','usam_social_media']], on='ano', how='left')
pew_interp = pew_interp.set_index('ano').interpolate(method='linear').reset_index()

# Extrapolação conservadora para 2011-2014 (Pew não tem dados teen antes de 2014-15)
# Baseada na curva de adoção de smartphones: 37% em 2013 → 73% em 2015 (Pew)
for year, oc, sm in [(2011,12,55),(2012,15,62),(2013,18,68),(2014,21,72)]:
    pew_interp.loc[pew_interp['ano']==year, 'online_constantly'] = oc
    pew_interp.loc[pew_interp['ano']==year, 'usam_social_media'] = sm
pew_interp = pew_interp.set_index('ano').interpolate(method='linear').reset_index()

# Combinar os dois eixos
dados = yrbs_interp.merge(pew_interp, on='ano', how='inner')
dados = dados[(dados['ano'] >= 2011) & (dados['ano'] <= 2023)]
dados['yrbs_real'] = dados['ano'].isin(yrbs['ano'].values)
dados['pew_real'] = dados['ano'].isin(pew['ano'].values)

print('Tabela-mestre unificada (R = dado real, I = interpolado):')
dados_display = dados.copy()
dados_display['YRBS'] = dados_display['yrbs_real'].map({True:'R', False:'I'})
dados_display['Pew'] = dados_display['pew_real'].map({True:'R', False:'I'})
display(dados_display[['ano','depressao_overall','depressao_feminino','depressao_masculino','online_constantly','YRBS','Pew']])

### 2.4 Gráfico 1 — Tendências temporais paralelas

Este gráfico é o ponto de partida visual da análise. Ele sobrepõe as duas séries temporais em eixos duplos para avaliar se as tendências se movem na mesma direção. Pontos sólidos representam dados reais; linhas tracejadas representam valores interpolados.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 7))

ax1.plot(dados['ano'], dados['depressao_overall'], color=COLORS['depression'], alpha=0.3, linewidth=1, linestyle='--')
yrbs_pts = dados[dados['yrbs_real']]
ax1.plot(yrbs_pts['ano'], yrbs_pts['depressao_overall'], color=COLORS['depression'], marker='o', markersize=8, linewidth=2.5, label='Depressão — YRBS (dados reais)', zorder=5)
for _, row in yrbs_pts.iterrows():
    ax1.annotate(f"{row['depressao_overall']:.1f}%", (row['ano'], row['depressao_overall']), textcoords='offset points', xytext=(0,12), fontsize=9, fontweight='bold', color=COLORS['depression'], ha='center')
ax1.set_xlabel('Ano', fontweight='bold')
ax1.set_ylabel('Tristeza/desesperança persistente (%)', color=COLORS['depression'], fontweight='bold')
ax1.tick_params(axis='y', labelcolor=COLORS['depression'])
ax1.set_ylim(10, 55)

ax2 = ax1.twinx()
ax2.plot(dados['ano'], dados['online_constantly'], color=COLORS['social_media'], alpha=0.3, linewidth=1, linestyle='--')
pew_pts = dados[dados['pew_real']]
ax2.plot(pew_pts['ano'], pew_pts['online_constantly'], color=COLORS['social_media'], marker='s', markersize=8, linewidth=2.5, label='Online constantemente — Pew (dados reais)', zorder=5)
for _, row in pew_pts.iterrows():
    ax2.annotate(f"{row['online_constantly']:.0f}%", (row['ano'], row['online_constantly']), textcoords='offset points', xytext=(0,-16), fontsize=9, fontweight='bold', color=COLORS['social_media'], ha='center')
ax2.set_ylabel('Online "quase constantemente" (%)', color=COLORS['social_media'], fontweight='bold')
ax2.tick_params(axis='y', labelcolor=COLORS['social_media'])
ax2.set_ylim(5, 60)

ax1.set_xticks(range(2011, 2024))
ax1.set_xticklabels(range(2011, 2024), rotation=45)
ax1.grid(True, alpha=0.3, color=COLORS['grid'])
ax1.axvspan(2020.5, 2021.5, alpha=0.08, color='gray', zorder=0)
ax1.text(2021, 12, 'COVID-19', ha='center', fontsize=8, color='gray', fontstyle='italic')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left', fontsize=9)
plt.title('Tendências Paralelas: Depressão Adolescente vs. Uso Constante de Redes Sociais\nEUA, 2011–2023', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 2.5 Análise de correlação

Calculamos dois coeficientes de correlação:
- **Pearson (r):** mede associação linear. Sensível a outliers.
- **Spearman (ρ):** mede associação monotônica (de ranking). Mais robusto para amostras pequenas.

Além da correlação principal, testamos: estratificação por sexo e robustez excluindo 2021 (efeito COVID-19).

In [ ]:
def correlacao(x, y, label):
    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)
    forca = 'FORTE' if abs(r_p)>=0.7 else ('MODERADA' if abs(r_p)>=0.3 else 'FRACA')
    sig = '✅ SIG.' if p_p < 0.05 else '⚠️ N.S.'
    print(f'  {label}')
    print(f'    Pearson  r = {r_p:+.4f}  p = {p_p:.6f}  R² = {r_p**2:.4f}  | {forca} | {sig}')
    print(f'    Spearman ρ = {r_s:+.4f}  p = {p_s:.6f}')
    print()
    return {'r': r_p, 'p': p_p, 'rho': r_s, 'p_s': p_s}

print('═' * 70)
print('  RESULTADOS DE CORRELAÇÃO — Parte A')
print('═' * 70)
print()
res_main = correlacao(dados['online_constantly'].values, dados['depressao_overall'].values, 'PRINCIPAL: Depressão overall × Online constantemente (n=13)')
res_f = correlacao(dados['online_constantly'].values, dados['depressao_feminino'].values, 'FEMININO: Depressão feminina × Online constantemente')
res_m = correlacao(dados['online_constantly'].values, dados['depressao_masculino'].values, 'MASCULINO: Depressão masculina × Online constantemente')
d_nc = dados[dados['ano'] != 2021]
res_rob = correlacao(d_nc['online_constantly'].values, d_nc['depressao_overall'].values, 'ROBUSTEZ: Sem 2021/COVID (n=12)')

### 2.6 Gráfico 2 — Scatter plot com regressão linear

Este gráfico testa visualmente a linearidade da relação. Cada ponto é um ano; a posição horizontal é o uso de redes sociais e a vertical é a depressão. A reta de regressão e o intervalo de confiança de 95% indicam a força e a precisão da associação. A caixa de texto resume os resultados estatísticos.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
x, y = dados['online_constantly'].values, dados['depressao_overall'].values
slope, intercept, r_val, p_val, se = stats.linregress(x, y)
x_line = np.linspace(x.min()-2, x.max()+2, 100)

n = len(x)
x_mean = np.mean(x)
se_line = se * np.sqrt(1/n + (x_line - x_mean)**2 / np.sum((x - x_mean)**2))
t_crit = stats.t.ppf(0.975, n-2)
ax.fill_between(x_line, slope*x_line+intercept - t_crit*se_line, slope*x_line+intercept + t_crit*se_line, alpha=0.15, color=COLORS['accent'], label='IC 95%')
ax.plot(x_line, slope*x_line+intercept, color=COLORS['accent'], linewidth=2, linestyle='--', label=f'Regressão (R² = {r_val**2:.3f})')

for _, row in dados.iterrows():
    c = COLORS['depression'] if (row['yrbs_real'] and row['pew_real']) else (COLORS['social_media'] if (row['yrbs_real'] or row['pew_real']) else '#999999')
    m = 'o' if (row['yrbs_real'] and row['pew_real']) else ('D' if (row['yrbs_real'] or row['pew_real']) else 'x')
    s = 100 if (row['yrbs_real'] and row['pew_real']) else 70
    ax.scatter(row['online_constantly'], row['depressao_overall'], c=c, marker=m, s=s, zorder=10, edgecolors='white', linewidth=0.5)
    ax.annotate(str(int(row['ano'])), (row['online_constantly'], row['depressao_overall']), textcoords='offset points', xytext=(8,5), fontsize=8, color='#555')

ax.text(0.98, 0.05, f'Pearson r = {r_val:+.4f} (p = {p_val:.4f})\nSpearman ρ = {res_main["rho"]:+.4f}\nR² = {r_val**2:.4f}\nn = {n}', transform=ax.transAxes, fontsize=9, va='bottom', ha='right', fontfamily='monospace', bbox=dict(boxstyle='round,pad=0.5', facecolor=COLORS['bg'], edgecolor=COLORS['grid']))
ax.set_xlabel('Online "quase constantemente" (%)', fontweight='bold')
ax.set_ylabel('Tristeza/desesperança persistente (%)', fontweight='bold')
ax.set_title('Correlação: Depressão × Uso de Redes Sociais — EUA, 2011–2023', fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3, color=COLORS['grid'])
plt.tight_layout()
plt.show()

### 2.7 Gráfico 3 — Estratificação por sexo

A literatura indica que o efeito das redes sociais na saúde mental pode diferir entre sexos. A meta-análise de Liu et al. (2022) encontrou OR = 1,72 para meninas vs. 1,20 para meninos. Testamos essa hipótese com nossos dados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for idx, (col, color, titulo) in enumerate([('depressao_feminino',COLORS['depression_f'],'Feminino'), ('depressao_masculino',COLORS['depression_m'],'Masculino')]):
    ax = axes[idx]
    xd, yd = dados['online_constantly'].values, dados[col].values
    sl, ic, r, p, _ = stats.linregress(xd, yd)
    rho, ps = stats.spearmanr(xd, yd)
    ax.scatter(xd, yd, c=color, s=80, zorder=5, edgecolors='white')
    xl = np.linspace(xd.min()-2, xd.max()+2, 100)
    ax.plot(xl, sl*xl+ic, color=color, linewidth=2, linestyle='--', alpha=0.7)
    for _, row in dados.iterrows():
        ax.annotate(str(int(row['ano'])), (row['online_constantly'], row[col]), textcoords='offset points', xytext=(6,4), fontsize=7, color='#777')
    ax.text(0.97, 0.05, f'r = {r:+.3f} (p={p:.4f})\nρ = {rho:+.3f}\nR² = {r**2:.3f}', transform=ax.transAxes, fontsize=9, va='bottom', ha='right', fontfamily='monospace', bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=COLORS['grid']))
    ax.set_title(titulo, fontweight='bold', color=color)
    ax.set_xlabel('Online "quase constantemente" (%)', fontsize=10)
    ax.set_ylabel('Tristeza/desesperança persistente (%)', fontsize=10)
    ax.grid(True, alpha=0.3, color=COLORS['grid'])
fig.suptitle('Estratificação por Sexo — EUA, 2011–2023', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.8 Gráfico 4 — Gap de gênero ao longo do tempo

Este gráfico destaca como a diferença entre meninas e meninos nos indicadores de depressão evoluiu ao longo do período. O crescimento do gap sugere que algum fator afeta desproporcionalmente as adolescentes — a literatura aponta redes sociais como um dos candidatos, mas não o único.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
bw = 0.35
xp = np.arange(len(yrbs))
b1 = ax.bar(xp-bw/2, yrbs['depressao_feminino'], bw, label='Feminino', color=COLORS['depression_f'], edgecolor='white')
b2 = ax.bar(xp+bw/2, yrbs['depressao_masculino'], bw, label='Masculino', color=COLORS['depression_m'], edgecolor='white')
for b, c in [(b1, COLORS['depression_f']),(b2, COLORS['depression_m'])]:
    for bar in b:
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.8, f'{bar.get_height():.1f}%', ha='center', fontsize=8, fontweight='bold', color=c)
for i, row in yrbs.iterrows():
    gap = row['depressao_feminino'] - row['depressao_masculino']
    ax.annotate(f'Δ {gap:.1f}pp', xy=(i, row['depressao_feminino']+3.5), fontsize=7, ha='center', color='#888', fontstyle='italic')
ax.set_xticks(xp)
ax.set_xticklabels(yrbs['ano'].astype(int))
ax.set_ylabel('Tristeza/desesperança persistente (%)', fontweight='bold')
ax.set_title('Gap de Gênero na Depressão Adolescente — YRBS, 2011–2023', fontweight='bold', pad=15)
ax.legend(loc='upper left')
ax.grid(True, axis='y', alpha=0.3, color=COLORS['grid'])
ax.set_ylim(0, 65)
plt.tight_layout()
plt.show()

---
## 3. PARTE B — Análise Intra-Survey (YRBS 2023)

O YRBS 2023 foi o primeiro ciclo a incluir uma pergunta sobre frequência de uso de redes sociais (Q87). Isso permite correlacionar uso de redes e depressão **nos mesmos indivíduos** (N = 20.103), eliminando a falácia ecológica da Parte A.

- **QN26** = tristeza/desesperança persistente (1 = sim, 2 = não)
- **Q87** = frequência de uso de redes sociais (escala de 7 pontos)
  - 1 = Mais de 1x por hora (mais frequente)
  - 2 = ~1x por hora
  - 3 = Várias vezes por dia
  - 4 = 1x por dia
  - 5 = 1–poucas vezes por semana
  - 6 = Poucas vezes por mês
  - 7 = Não usa redes sociais (menos frequente)
- **QN87** = dicotômica CDC (1 = infrequente/não usa [Q87=1,2], 2 = uso semanal+ [Q87=3-7])
- **q2** = sexo (1 = feminino, 2 = masculino)

> **Nota importante:** A codificação do Q87 é reversa — valores menores indicam USO MAIS FREQUENTE. A análise abaixo utiliza os valores brutos de Q87 para a análise dose-resposta.

### 3.1 Carregamento dos microdados

Os microdados são baixados automaticamente do repositório GitHub do projeto. O arquivo original (formato MDB do CDC) foi convertido para CSV.

In [ ]:
USING_REAL = False
csv_file = 'yrbs2023.csv'
csv_raw = 'yrbs2023_raw.csv'

# Download dichotomous file
if not os.path.exists(csv_file):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/rodrigobfferraz/mvp-depressao-e-redes-sociais/main/data/yrbs2023.csv', csv_file)

# Download raw file (for Q87 dose-response)
if not os.path.exists(csv_raw):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/rodrigobfferraz/mvp-depressao-e-redes-sociais/main/data/yrbs2023_raw.csv', csv_raw)

if os.path.exists(csv_file):
    df_dic = pd.read_csv(csv_file, low_memory=False)
    df_dic.columns = df_dic.columns.str.strip().str.lower()
    USING_REAL = True
    print(f'✅ Dados reais carregados: {len(df_dic)} registros')
else:
    print('⚠️  Microdados não encontrados. Usando dados simulados.')
    np.random.seed(2023)
    N = 20103
    sex = np.random.choice([1,2], size=N, p=[0.50,0.50])
    p_freq = np.where(sex==1, 0.80, 0.74)
    freq_sm = np.array([np.random.choice([1,2], p=[p,1-p]) for p in p_freq])
    p_dep = np.where(sex==1, np.where(freq_sm==1, 0.57, 0.38), np.where(freq_sm==1, 0.31, 0.20))
    depression = np.array([np.random.choice([1,2], p=[p,1-p]) for p in p_dep])
    df_dic = pd.DataFrame({'q2':sex, 'qn26':depression, 'qn87':freq_sm, 'weight':np.ones(N)})

# Load raw data for dose-response analysis
if os.path.exists(csv_raw):
    df_raw = pd.read_csv(csv_raw, low_memory=False)
    df_raw.columns = df_raw.columns.str.strip().str.lower()
    has_raw = True
else:
    has_raw = False

# Create analysis variables
df = df_dic.copy()
df['depressao'] = (df['qn26']==1).astype(int)  # QN26=1 = YES depression

# Add raw Q87 from raw file if available
if has_raw:
    df['q87_raw'] = df_raw['q87'].values
    # CDC MMWR definition: frequent = 'at least several times a day' = Q87 <= 3
    # (Q87: 1=most frequent, 7=no SM)
    df['uso_freq_sm'] = (df['q87_raw'].isin([1,2,3])).astype(int)
else:
    # Fallback to QN87 (note: QN87=1 means INFREQUENT in this dataset)
    df['uso_freq_sm'] = (df['qn87']==2).astype(int)

col_sex = 'q2'
df['sexo'] = df['q2'].map({1:'Feminino',2:'Masculino'})
df_clean = df.dropna(subset=['depressao','uso_freq_sm'])
data_lbl = 'REAIS' if USING_REAL else 'SIMULADOS'
print(f'Registros válidos: {len(df_clean):,} | Dados: {data_lbl}')

### 3.2 Análise Exploratória dos Microdados (EDA — Parte B)

Antes da análise de associação, inspecionamos a distribuição das variáveis-chave nos microdados individuais.

In [ ]:
print('═' * 60)
print(f'  EDA — MICRODADOS YRBS 2023 (N = {len(df_clean):,}) [{data_lbl}]')
print('═' * 60)

prev_dep = df_clean['depressao'].mean() * 100
prev_sm = df_clean['uso_freq_sm'].mean() * 100
print(f'\n  Prevalências:')
print(f'    Depressão (tristeza/desesperança): {prev_dep:.1f}%')
print(f'    Uso frequente de SM (≥ várias vezes/dia): {prev_sm:.1f}%')

print(f'\n  Distribuição por sexo:')
for s in ['Feminino', 'Masculino']:
    sub = df_clean[df_clean['sexo']==s]
    print(f'    {s}: n={len(sub):,} | dep={sub["depressao"].mean()*100:.1f}% | SM freq={sub["uso_freq_sm"].mean()*100:.1f}%')

# Dose-response if raw Q87 available
if has_raw:
    labels = {1:'Mais de 1x/hora', 2:'~1x por hora', 3:'Várias vezes/dia',
              4:'1x por dia', 5:'1-poucas vezes/semana', 6:'Poucas vezes/mês', 7:'Não usa SM'}
    print(f'\n  Dose-resposta (prevalência de depressão por frequência de uso):')
    for val in [1,2,3,4,5,6,7]:
        sub = df_clean[(df_clean['q87_raw']==val) & df_clean['depressao'].notna()]
        if len(sub) > 0:
            wt = sub['weight'] if 'weight' in sub.columns else None
            dep = np.average(sub['depressao'], weights=wt)*100 if wt is not None else sub['depressao'].mean()*100
            print(f'    {labels[val]:25s}: {dep:.1f}% (n={len(sub)})')

### 3.3 Tabela cruzada e testes estatísticos

A análise central da Parte B: cruzar uso frequente de redes sociais com depressão e testar a significância da associação. Calculamos:
- **Qui-quadrado (χ²):** testa independência entre as variáveis
- **Razão de Prevalência (PR):** quantas vezes mais prevalente é a depressão entre quem usa SM frequentemente
- **Odds Ratio (OR):** chance relativa de depressão, com intervalo de confiança 95%

In [ ]:
freq_u = df_clean[df_clean['uso_freq_sm']==1]
infreq_u = df_clean[df_clean['uso_freq_sm']==0]

dep_freq = freq_u['depressao'].mean()*100
dep_infreq = infreq_u['depressao'].mean()*100

ct = pd.crosstab(df_clean['uso_freq_sm'], df_clean['depressao'])
chi2, p_chi2, dof, _ = stats.chi2_contingency(ct)

PR = dep_freq / dep_infreq if dep_infreq > 0 else 0
a,b,c,d = ct.iloc[1,1], ct.iloc[1,0], ct.iloc[0,1], ct.iloc[0,0]
OR = (a*d)/(b*c) if (b*c) > 0 else 0
se_ln = np.sqrt(1/a+1/b+1/c+1/d) if min(a,b,c,d) > 0 else 0
ci_lo = np.exp(np.log(OR)-1.96*se_ln) if OR > 0 else 0
ci_hi = np.exp(np.log(OR)+1.96*se_ln) if OR > 0 else 0

print(f'═' * 60)
print(f'  RESULTADOS INTRA-SURVEY — YRBS 2023 [{data_lbl}]')
print(f'═' * 60)
print(f'\n  Uso frequente = várias vezes ao dia ou mais (Q87 ≤ 3)')
print(f'  Depressão com uso frequente de SM:    {dep_freq:.1f}% (n={len(freq_u):,})')
print(f'  Depressão sem uso frequente de SM:     {dep_infreq:.1f}% (n={len(infreq_u):,})')
print(f'  Diferença: {dep_freq-dep_infreq:+.1f} pontos percentuais')
print(f'\n  χ² = {chi2:.1f} (p = {p_chi2:.2e})')
print(f'  PR = {PR:.2f}')
print(f'  OR = {OR:.2f} (IC 95%: {ci_lo:.2f}–{ci_hi:.2f})')

if col_sex:
    print(f'\n  Estratificação por sexo:')
    for sx, lb in [(1,'Feminino'),(2,'Masculino')]:
        sub = df_clean[df_clean[col_sex]==sx]
        f_sub = sub[sub['uso_freq_sm']==1]['depressao'].mean()*100
        i_sub = sub[sub['uso_freq_sm']==0]['depressao'].mean()*100
        ct_sub = pd.crosstab(sub['uso_freq_sm'], sub['depressao'])
        chi2_s, p_s, _, _ = stats.chi2_contingency(ct_sub)
        pr_s = f_sub / i_sub if i_sub > 0 else 0
        print(f'    {lb}: dep. freq={f_sub:.1f}% | dep. infreq={i_sub:.1f}% | Δ={f_sub-i_sub:+.1f}pp | PR={pr_s:.2f} | χ²={chi2_s:.1f} (p={p_s:.2e})')

### 3.4 Gráfico 5 — Depressão por uso de redes sociais e sexo

Visualização direta da associação: comparação das taxas de depressão entre quem usa redes sociais frequentemente e quem não usa, no geral e estratificado por sexo.

In [ ]:
cats, fv, iv = ['Overall'], [dep_freq], [dep_infreq]
if col_sex:
    for sx, lb in [(1,'Feminino'),(2,'Masculino')]:
        sub = df_clean[df_clean[col_sex]==sx]
        cats.append(lb)
        fv.append(sub[sub['uso_freq_sm']==1]['depressao'].mean()*100)
        iv.append(sub[sub['uso_freq_sm']==0]['depressao'].mean()*100)

fig, ax = plt.subplots(figsize=(10,7))
xp = np.arange(len(cats)); bw = 0.35
b1 = ax.bar(xp-bw/2, fv, bw, label='Uso frequente de SM', color=COLORS['freq_sm'], edgecolor='white')
b2 = ax.bar(xp+bw/2, iv, bw, label='Uso infrequente / Não usa', color=COLORS['infreq_sm'], edgecolor='white')
for bars, c in [(b1,COLORS['freq_sm']),(b2,COLORS['infreq_sm'])]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.8, f'{bar.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold', color=c)
for i in range(len(cats)):
    ax.annotate(f'Δ {fv[i]-iv[i]:+.1f}pp', xy=(i, max(fv[i],iv[i])+4), ha='center', fontsize=9, color='#555', fontstyle='italic', fontweight='bold')
ax.set_xticks(xp); ax.set_xticklabels(cats, fontsize=12)
ax.set_ylabel('Tristeza/desesperança persistente (%)', fontweight='bold')
ax.set_title(f'Depressão por Frequência de Uso de SM — YRBS 2023 (N={len(df_clean):,}) [{data_lbl}]', fontweight='bold', pad=15)
ax.legend(loc='upper right')
ax.text(0.02, 0.97, f'χ² = {chi2:.1f} (p < 0.001)\nPR = {PR:.2f}\nOR = {OR:.2f} (IC 95%: {ci_lo:.2f}–{ci_hi:.2f})', transform=ax.transAxes, fontsize=9, va='top', fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLORS['grid']))
ax.grid(True, axis='y', alpha=0.3, color=COLORS['grid'])
ax.set_ylim(0, max(fv)+12)
plt.tight_layout()
plt.show()

### 3.5 Gráfico 6 — Heatmap por sexo e uso de SM

Mapa de calor resumindo a prevalência de depressão nas quatro combinações: feminino/masculino × uso frequente/infrequente. A cor indica intensidade — quanto mais escuro, maior a prevalência.

In [ ]:
if col_sex and len(cats) > 1:
    hm = pd.DataFrame({'Uso frequente': fv[1:], 'Uso infrequente': iv[1:]}, index=cats[1:])
    fig, ax = plt.subplots(figsize=(8,5))
    sns.heatmap(hm, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=2, linecolor='white',
                annot_kws={'size':16,'fontweight':'bold'}, cbar_kws={'label':'% com depressão'},
                vmin=15, vmax=60, ax=ax)
    ax.set_title(f'Prevalência de Depressão — Sexo × Uso de SM — YRBS 2023 [{data_lbl}]', fontweight='bold', pad=15)
    plt.tight_layout()
    plt.show()

### 3.6 Gráfico 7 — Dose-resposta: depressão por nível de frequência de uso de SM

Este gráfico é o achado mais importante da Parte B: a relação entre frequência de uso de redes sociais e depressão NÃO é linear. Os usuários mais intensivos têm menor depressão do que os de frequência intermediária. Isso demonstra empiricamente a falácia ecológica e é consistente com a hipótese Goldilocks (Przybylski & Weinstein, 2017).

In [ ]:
if has_raw:
    labels = {1:'> 1x/hora', 2:'~1x/hora', 3:'Várias/dia',
              4:'1x/dia', 5:'1-poucas/sem', 6:'Poucas/mês', 7:'Não usa'}
    vals, deps, ns = [], [], []
    for v in [1,2,3,4,5,6,7]:
        sub = df_clean[(df_clean['q87_raw']==v) & df_clean['depressao'].notna()]
        if len(sub) > 0:
            wt = sub['weight'] if 'weight' in sub.columns else None
            dep = np.average(sub['depressao'], weights=wt)*100 if wt is not None else sub['depressao'].mean()*100
            vals.append(labels[v])
            deps.append(dep)
            ns.append(len(sub))

    fig, ax = plt.subplots(figsize=(12, 7))
    colors = ['#27AE60','#2ECC71','#F39C12','#E67E22','#E74C3C','#95A5A6','#7F8C8D']
    bars = ax.bar(range(len(vals)), deps, color=colors, edgecolor='white', linewidth=0.5)
    for i, (bar, dep, n) in enumerate(zip(bars, deps, ns)):
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+1, f'{dep:.1f}%', ha='center', fontsize=10, fontweight='bold', color=colors[i])
        ax.text(bar.get_x()+bar.get_width()/2., 2, f'n={n:,}', ha='center', fontsize=8, color='white', fontweight='bold')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals, fontsize=10, rotation=20, ha='right')
    ax.set_ylabel('Depressão — tristeza/desesperança persistente (%)', fontweight='bold')
    ax.set_title(f'Dose-Resposta: Depressão por Frequência de Uso de Redes Sociais\nYRBS 2023 (N={len(df_clean):,}) — Padrão Não Linear [{data_lbl}]', fontweight='bold', pad=15)
    ax.set_ylim(0, max(deps)+8)
    ax.grid(True, axis='y', alpha=0.3, color=COLORS['grid'])
    ax.annotate('← Uso mais frequente          Uso menos frequente →', xy=(0.5, -0.15), xycoords='axes fraction', ha='center', fontsize=10, color='#888', fontstyle='italic')
    plt.tight_layout()
    plt.show()
else:
    print('Dados brutos Q87 não disponíveis — gráfico dose-resposta não gerado.')

---
## 4. Síntese e Conclusões

In [ ]:
print(f'''
{'═'*70}
  SÍNTESE DOS RESULTADOS
{'═'*70}

  PARTE A — Séries temporais (2011–2023, dados agregados):
    Pearson  r = {res_main['r']:+.4f}  (p = {res_main['p']:.6f})
    Spearman ρ = {res_main['rho']:+.4f}
    R²         = {res_main['r']**2:.4f} ({res_main['r']**2*100:.1f}% da variância)
    Feminino:  r = {res_f['r']:+.4f}   Masculino: r = {res_m['r']:+.4f}
    Robustez (sem COVID): r = {res_rob['r']:+.4f}

  PARTE B — Intra-survey YRBS 2023 (dados individuais) [{data_lbl}]:
    Depressão com uso freq. SM (≥ várias vezes/dia): {dep_freq:.1f}%
    Depressão sem uso freq. SM: {dep_infreq:.1f}%
    OR = {OR:.2f} (IC 95%: {ci_lo:.2f}–{ci_hi:.2f})
    χ² = {chi2:.1f} (p = {p_chi2:.2e})

  ACHADO PRINCIPAL:
    A análise dose-resposta revelou padrão NÃO LINEAR:
    uso mais intensivo (>1x/hora) = MENOR depressão (31.6%)
    uso intermediário (1x/dia-1x/sem) = MAIOR depressão (55-60%)
    não usuários = nível intermediário (42.4%)
    → Demonstra empiricamente a falácia ecológica

  CONCLUSÃO:
    H₀ rejeitada na Parte A (correlação ecológica forte).
    Parte B revela que a relação individual é mais complexa
    do que a correlação populacional sugere.

  LIMITAÇÕES:
    1. Correlação ≠ causalidade
    2. Falácia ecológica demonstrada (Parte A vs B)
    3. Cross-sectional na Parte B
    4. Dados auto-reportados
    5. Interpolação linear na Parte A
    6. Sem controle de variáveis confundidoras
{'═'*70}
''')

### Referências

1. Centers for Disease Control and Prevention. *Youth Risk Behavior Survey Data Summary & Trends Report: 2013–2023.* U.S. DHHS; 2024.
2. Pew Research Center. *Teens, Social Media and Technology 2024.* Washington, DC; 2024.
3. Pew Research Center. *Teens, Social Media and Technology 2023.* Washington, DC; 2023.
4. Pew Research Center. *Teens, Social Media & Technology 2018.* Washington, DC; 2018.
5. Kreski, N. et al. Social Media Use and Depressive Symptoms Among United States Adolescents. *J Adolesc Health*, 68(3), 572-579, 2021.
6. Liu, M. et al. Time Spent on Social Media and Risk of Depression in Adolescents: A Dose-Response Meta-Analysis. *Int. J. Environ. Res. Public Health*, 19(9), 5164, 2022.
7. Office of the Surgeon General. *Social Media and Youth Mental Health: The U.S. Surgeon General's Advisory.* Washington, DC; 2023.
8. Rideout, V. et al. *The Common Sense Census: Media Use by Tweens and Teens, 2021.* Common Sense Media; 2022.
9. SAMHSA. *Key Substance Use and Mental Health Indicators: Results from the 2021 NSDUH.* Rockville, MD; 2022.

---

*Notebook gerado como parte do MVP — MVP — Minimum Viable Product.*
*Repositório: [github.com/rodrigobfferraz/mvp-depressao-e-redes-sociais](https://github.com/rodrigobfferraz/mvp-depressao-e-redes-sociais)*